# RAG Assignment: Baseline vs. Re-Ranking
**Document:** FedRAMP Continuous Monitoring (ConMon) Playbook  
**LLM:** Groq API (`qwen/qwen3.8-27b`)  
**Embeddings:** `all-MiniLM-L6-v2` (Bi-Encoder, FAISS)  
**Re-Ranker:** `cross-encoder/ms-marco-MiniLM-L-6-v2`  

---
**Pipeline 1 (Baseline):** embed query → FAISS Top-10 → Top-3 → Groq Qwen 3.8  
**Pipeline 2 (Re-Ranked):** same Top-10 → Cross-Encoder → re-sort → Top-3 → Groq Qwen 3.8

## 0. Install Dependencies

In [34]:
!pip install -q pymupdf sentence-transformers faiss-cpu groq numpy pandas

## 1. Setup & Imports

In [35]:
import json, os, time, warnings
import faiss
import numpy as np
from groq import Groq 
import pandas as pd
import pymupdf
from sentence_transformers import CrossEncoder, SentenceTransformer

warnings.filterwarnings('ignore')
print('All imports successful.')

All imports successful.


In [36]:
# ─── CONFIGURATION ────────────────────────────────────────────────────────
GROQ_API_KEY = (
    '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'  
)
PDF_PATH = '/kaggle/input/datasets/giridharanp/continuous-monitoring-playbook/Continuous_Monitoring_Playbook.pdf'
EMBED_MODEL_NAME = 'all-MiniLM-L6-v2'
CROSS_ENC_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
LLM_MODEL_NAME = 'qwen/qwen3.8-27b'  # Standard operational Groq model ID
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
TOP_K_INITIAL = 10
TOP_K_FINAL = 3

client = Groq(api_key=GROQ_API_KEY)
print(f'Groq client configured for {LLM_MODEL_NAME}')

Groq client configured for qwen/qwen3.8-27b


## 2. Load & Chunk the Document

In [37]:
import fitz  # PyMuPDF - PDF parsing


def extract_text_from_pdf(path):
    """Extract all text pages from the PDF using PyMuPDF."""
    doc = fitz.open(path)
    pages = [page.get_text('text') for page in doc]
    doc.close()
    return '\n'.join(pages)


def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Sliding-window character-level chunking.

    Tries to snap the chunk boundary to a sentence-ending period so chunks
    don't cut mid-sentence.
    """
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end]
        if end < len(text):
            lp = chunk.rfind('. ')
            if lp > chunk_size // 2:
                end = start + lp + 2
                chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk.strip())

        # Exit loop once the end of text is reached
        if end == len(text):
            break

        start = end - overlap
    return chunks


print(f'Loading: {PDF_PATH}')
full_text = extract_text_from_pdf(PDF_PATH)
print(f'Characters extracted: {len(full_text):,}')
chunks = chunk_text(full_text)
print(f'Chunks created:       {len(chunks):,}')
print('\nSample chunk #42:\n' + '-' * 60)
print(chunks[42])

Loading: /kaggle/input/datasets/giridharanp/continuous-monitoring-playbook/Continuous_Monitoring_Playbook.pdf
Characters extracted: 82,101
Chunks created:       251

Sample chunk #42:
------------------------------------------------------------
reference number must be included with the machine-readable findings data for 
that vulnerability. 
Common Vulnerability Scoring System (CVSS) Risk Scoring: For any vulnerability with a CVSSv3 
base score assigned in the latest version of the NVD, the CVSSv3 base score must be used as the 
original risk rating. If no CVSSv3 score is available, a CVSSv2 base score is acceptable where 
available. If no CVSS score is available, the native scanner base risk score can be used.


## 3. Build the Vector Index (Bi-Encoder + FAISS)

In [38]:
print(f'Loading bi-encoder: {EMBED_MODEL_NAME}')
bi_encoder = SentenceTransformer(EMBED_MODEL_NAME)

print('Encoding chunks (may take ~1-2 min)...')
t0 = time.time()
# normalize_embeddings=True so inner-product == cosine similarity
chunk_embeddings = bi_encoder.encode(
    chunks, batch_size=64, show_progress_bar=True, normalize_embeddings=True
)
print(f'Done in {time.time()-t0:.1f}s  |  shape: {chunk_embeddings.shape}')

Loading bi-encoder: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding chunks (may take ~1-2 min)...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Done in 7.8s  |  shape: (251, 384)


In [39]:
# FAISS IndexFlatIP = exact inner-product (== cosine for L2-normalised vectors)
dim = chunk_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(chunk_embeddings.astype(np.float32))
print(f'FAISS index: {faiss_index.ntotal} vectors, dim={dim}')

FAISS index: 251 vectors, dim=384


## 4. Load the Cross-Encoder

In [40]:
print(f'Loading cross-encoder: {CROSS_ENC_NAME}')
cross_encoder = CrossEncoder(CROSS_ENC_NAME)
print('Cross-encoder ready.')

Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-encoder ready.


## 5. Core Pipeline Functions

In [47]:
def baseline_retrieve(
    query, top_k_initial=TOP_K_INITIAL, top_k_final=TOP_K_FINAL
):
    """Pipeline 1 — Baseline Retrieval:

    1. Embed query with Bi-Encoder. 2. Search FAISS for top_k_initial chunks by
    cosine similarity. 3. Return top_k_final chunks directly (NO re-ranking).
    Returns: (top_chunks: list[str], all_results: list[dict], latency_sec:
    float)
    """
    t0 = time.time()
    q_emb = bi_encoder.encode([query], normalize_embeddings=True).astype(
        np.float32
    )
    scores, idxs = faiss_index.search(q_emb, top_k_initial)
    all_results = [
        {
            'baseline_rank': r + 1,
            'chunk_id': int(i),
            'cosine_score': float(s),
            'text': chunks[i],
        }
        for r, (i, s) in enumerate(zip(idxs[0], scores[0]))
    ]
    return (
        [x['text'] for x in all_results[:top_k_final]],
        all_results,
        time.time() - t0,
    )


def reranked_retrieve(
    query, top_k_initial=TOP_K_INITIAL, top_k_final=TOP_K_FINAL
):
    """Pipeline 2 — Cross-Encoder Re-Ranking:

    1. Same Bi-Encoder retrieval of top_k_initial candidates. 2. Score ALL
    (query, chunk) pairs jointly with the Cross-Encoder.
       The Cross-Encoder sees both sequences together and attends across every
       token pair, capturing negation / synonyms / context.
    3. Re-sort by CE score descending; select top_k_final.
    Returns: (top_chunks: list[str], reranked: list[dict], latency_sec: float)
    """
    t0 = time.time()
    q_emb = bi_encoder.encode([query], normalize_embeddings=True).astype(
        np.float32
    )
    scores, idxs = faiss_index.search(q_emb, top_k_initial)
    candidates = [
        {
            'baseline_rank': r + 1,
            'chunk_id': int(i),
            'cosine_score': float(s),
            'text': chunks[i],
        }
        for r, (i, s) in enumerate(zip(idxs[0], scores[0]))
    ]
    ce_scores = cross_encoder.predict([(query, c['text']) for c in candidates])
    for c, sc in zip(candidates, ce_scores):
        c['ce_score'] = float(sc)
    reranked = sorted(candidates, key=lambda x: x['ce_score'], reverse=True)
    for i, x in enumerate(reranked):
        x['reranked_rank'] = i + 1
    return (
        [x['text'] for x in reranked[:top_k_final]],
        reranked,
        time.time() - t0,
    )


def build_prompt(query, context_chunks):
  context = '\n\n---\n\n'.join(
      [f'[Chunk {i+1}]\n{c}' for i, c in enumerate(context_chunks)]
  )
  return (
      'You are a precise compliance officer and technical documentation expert.\n'
      'Answer the supported part of the question using ONLY the provided context'
      ' chunks; explicitly state what information is missing if the prompt is'
      ' only partially answered.\n'
      "Say 'Not found in provided context.' ONLY if nothing relevant is present"
      ' in the context.\n'
      'Be concise and cite which chunk(s) support your answer.\n\n'
      f'CONTEXT:\n{context}\n\nQUESTION: {query}\n\nANSWER:'
  )

def generate_answer(query, context_chunks):
  """Call Groq API with the RAG prompt."""
  response = client.chat.completions.create(
      model=LLM_MODEL_NAME,
      messages=[
          {
              'role': 'system',
              'content': (
                  'You are a precise FedRAMP continuous monitoring and'
                  ' cybersecurity compliance expert.'
              ),
          },
          {'role': 'user', 'content': build_prompt(query, context_chunks)},
      ],
      temperature=0.0,
      max_tokens=256,  # <-- FIX: Caps expected output so Groq's OTPM pre-check passes
  )
  return response.choices[0].message.content.strip()
print('Pipeline functions defined.')

Pipeline functions defined.


## 6. Full RAG Comparison Runner

In [42]:
def run_comparison(query, query_label='', verbose=True):
    """Run both pipelines for a single query; print a side-by-side comparison table."""
    sep = '='*70
    print(f'\n{sep}')
    print(f'  [{query_label}]  Q: {query}')
    print(sep)

    # Pipeline 1 - Baseline
    b_chunks,  b_results,  b_lat  = baseline_retrieve(query)
    b_answer  = generate_answer(query, b_chunks)

    # Pipeline 2 - Re-Ranked
    rr_chunks, rr_results, rr_lat = reranked_retrieve(query)
    rr_answer = generate_answer(query, rr_chunks)

    if verbose:
        rows = []
        for b in b_results:
            rr = next((r for r in rr_results if r['chunk_id'] == b['chunk_id']), None)
            rr_rank = rr['reranked_rank'] if rr else '-'
            ce_sc   = f"{rr['ce_score']:.3f}" if rr else '-'
            shift   = (b['baseline_rank'] - rr_rank) if isinstance(rr_rank, int) else 0
            rows.append([
                b['baseline_rank'], rr_rank,
                f'+{shift}' if shift > 0 else str(shift),
                f"{b['cosine_score']:.3f}", ce_sc,
                (b['text'][:75] + '...').replace('\n', ' ')
            ])
        df = pd.DataFrame(rows,
            columns=['Baseline', 'Re-Rank', 'Shift', 'Cosine', 'CE Score', 'Preview'])
        print('\nRETRIEVAL TABLE (Top-10):')
        print(df.to_string(index=False))
        print(f'\n--- PIPELINE 1 (Baseline) ---\n{b_answer}')
        print(f'\n--- PIPELINE 2 (Re-Ranked) ---\n{rr_answer}')
        print(f'\nLatency:  Baseline={b_lat*1000:.1f}ms  '
              f'Re-Ranking={rr_lat*1000:.1f}ms  '
              f'Overhead=+{(rr_lat-b_lat)*1000:.1f}ms')

    return dict(
        query_label=query_label, query=query,
        baseline_results=b_results, reranked_results=rr_results,
        baseline_answer=b_answer, reranked_answer=rr_answer,
        baseline_latency_ms=b_lat*1000, reranked_latency_ms=rr_lat*1000
    )
def gold_rank(result, needle):
    needle = ' '.join(needle.lower().split())
    for r in result['baseline_results']:
        if needle in ' '.join(r['text'].lower().split()):
            rr = next(x for x in result['reranked_results'] if x['chunk_id'] == r['chunk_id'])
            print(f"Found '{needle}': Baseline #{r['baseline_rank']} -> Re-Ranked #{rr['reranked_rank']}")
            return
    print(f"Phrase '{needle}' not in Top-10 candidates.")
print('run_comparison() ready.')

run_comparison() ready.


## 7. Baseline Demo Queries

In [43]:
general_results = []
for q, lbl in [
    (
        'What are the core requirements and objectives of the FedRAMP Continuous Monitoring (ConMon) program?',
        'General-1: ConMon Overview & Objectives',
    ),
    (
        'How does the FedRAMP Playbook define a significant change versus a minor change?',
        'General-2: Significant Change Definition',
    ),
]:
    general_results.append(run_comparison(q, lbl))


  [General-1: ConMon Overview & Objectives]  Q: What are the core requirements and objectives of the FedRAMP Continuous Monitoring (ConMon) program?

RETRIEVAL TABLE (Top-10):
 Baseline  Re-Rank Shift Cosine CE Score                                                                        Preview
        1        6    -5  0.825    3.395 5.0 (2024)  ●​ FedRAMP Collaborative ConMon Quick Guide (2023)  ●​ FedRAMP ...
        2        3    -1  0.748    4.133 laborative ConMon Charter​ 29  Step 4: Hold Monthly Recurring Collaborative...
        3        1    +2  0.737    5.471 ing a plan of action and milestones (POA&Ms) and/or CAPs to  address areas ...
        4        2    +2  0.726    5.462 book provides an overview of FedRAMP Rev 5  continuous monitoring (ConMon) ...
        5        8    -3  0.712    3.055 on  FedRAMP      fedramp.gov           ConMon Playbook    TABLE OF CONTENTS...
        6        4    +2  0.628    3.924 ation” status) when the CSP fails  to comply with FedRAMP’s co

## 8. Failure Mode Analysis — Stress Testing

We evaluate **4 failure modes** where the Bi-Encoder baseline degrades 
and the Cross-Encoder re-ranker successfully recovers relevant chunks:

| # | Failure Type | Description |
|---|---|---|
| **F1** | Keyword Trap / Distractor | High lexical overlap with irrelevant passages |
| **F2** | Exceptions & Corner Cases | Baseline returns general rule; misses the `except when…` clause |
| **F3** | Negation & Constraints | `prohibited`/`not allowed` queries; bi-encoder blurs polarity |
| **F4** | Buried Gold (Rank 4–10) | Critical chunk sits outside Top-3; CE promotes it |

In [48]:
# ── FAILURE MODE 1: Keyword Trap / Distractor ────────────────────────────
f1_result = run_comparison(
    'How soon after identification must a CSP report an incident?',
    'F1: Keyword Trap — Incident Reporting Timelines',
)
gold_rank(f1_result, 'within one (1) hour')

# ── FAILURE MODE 2: Exception & Corner Cases ─────────────────────────────
f2_result = run_comparison(
    'Can a significant change be approved without a full Security Assessment Plan (SAP) and Security Assessment Report (SAR)?',
    'F2: Exception — Attestation Letter for Small Adaptive Changes',
)
gold_rank(f2_result, 'attestation')

# ── FAILURE MODE 3: Negation & Scope Constraints ─────────────────────────
f3_result = run_comparison(
    'Is the initial authorizing agency responsible for Continuous Monitoring (ConMon) oversight for other agencies?',
    'F3: Negation — Agency ConMon Oversight Responsibilities',
)
gold_rank(f3_result, 'is not responsible for performing')

# ── FAILURE MODE 4: Buried Gold (Needle in a Haystack) ────────────────────
f4_result = run_comparison(
    'At what age do five or more late high-impact vulnerabilities move from a DFR to a CAP?',
    'F4: Buried Gold — Table Threshold (60-Day Rule)',
)
gold_rank(f4_result, 'aged greater than 60 days')


  [F1: Keyword Trap — Incident Reporting Timelines]  Q: How soon after identification must a CSP report an incident?

RETRIEVAL TABLE (Top-10):
 Baseline  Re-Rank Shift Cosine CE Score                                                                        Preview
        1        7    -6  0.688    1.066 any emergency inquiry is  issued, the CSP must comply within the timeline d...
        2        1    +1  0.648    7.284 cessed, or transmitted by the  system as well as data about the system and ...
        3        4    -1  0.637    3.484 g  Incident Notifications section of https://www.cisa.gov/federal-incident-...
        4        8    -4  0.625    0.892 e.g., by phone) must be followed up by an email; however,  sensitive inform...
        5        5     0  0.623    3.208 to all POCs after it has completed the  Post-Incident Activity in the Incid...
        6        9    -3  0.610   -1.253 enter (SOC), or  information technology department to the following stakeho...
        7      

## 9. Rank-Shift Summary

In [49]:
def rank_shift_summary(result, top_k=TOP_K_FINAL):
    """Report which chunks were promoted into / demoted out of Top-3."""
    lbl   = result['query_label']
    b_ids  = {r['chunk_id'] for r in result['baseline_results'][:top_k]}
    rr_ids = {r['chunk_id'] for r in result['reranked_results'][:top_k]}
    promoted = rr_ids - b_ids
    demoted  = b_ids  - rr_ids

    print(f'\n--- {lbl} ---')
    if not promoted:
        print('  No change in Top-3 — baseline and re-ranked agree.')
        return
    for cid in promoted:
        b  = next(r for r in result['baseline_results'] if r['chunk_id']==cid)
        rr = next(r for r in result['reranked_results']  if r['chunk_id']==cid)
        print(f"  PROMOTED  Chunk #{cid}: Rank #{b['baseline_rank']} -> #{rr['reranked_rank']}  "
              f"(cosine={b['cosine_score']:.3f}, CE={rr['ce_score']:.3f})")
        print(f"  Preview: {rr['text'][:130].replace(chr(10), ' ')}...")
    for cid in demoted:
        b  = next(r for r in result['baseline_results'] if r['chunk_id']==cid)
        rr = next(r for r in result['reranked_results']  if r['chunk_id']==cid)
        print(f"  DEMOTED   Chunk #{cid}: Rank #{b['baseline_rank']} -> #{rr['reranked_rank']}  "
              f"(cosine={b['cosine_score']:.3f}, CE={rr['ce_score']:.3f})")
        print(f"  Preview: {b['text'][:130].replace(chr(10), ' ')}...")

print('='*70)
print('         RANK-SHIFT SUMMARY')
print('='*70)
for r in general_results + [f1_result, f2_result, f3_result, f4_result]:
  rank_shift_summary(r)

         RANK-SHIFT SUMMARY

--- General-1: ConMon Overview & Objectives ---
  PROMOTED  Chunk #8: Rank #4 -> #2  (cosine=0.726, CE=5.462)
  Preview: book provides an overview of FedRAMP Rev 5  continuous monitoring (ConMon) requirements and activities, along with guidance and be...
  DEMOTED   Chunk #11: Rank #1 -> #6  (cosine=0.825, CE=3.395)
  Preview: 5.0 (2024)  ●​ FedRAMP Collaborative ConMon Quick Guide (2023)  ●​ FedRAMP Continuous Monitoring Performance Management Guide, ver...

--- General-2: Significant Change Definition ---
  PROMOTED  Chunk #132: Rank #4 -> #3  (cosine=0.543, CE=2.387)
  Preview: the change is routine recurring in nature, the CSP proceeds  with normal monthly ConMon activities. If they determine the change i...
  DEMOTED   Chunk #0: Rank #3 -> #9  (cosine=0.568, CE=-3.995)
  Preview: FedRAMP® Continuous  Monitoring Playbook  Version 1.0  11/17/2025              info@fedramp.gov  FedRAMP.gov         ConMon Playbo...

--- F1: Keyword Trap — Incident Reportin

## 10. Latency Profiling

In [50]:
all_results = general_results + [f1_result, f2_result, f3_result, f4_result]

rows = []
for r in all_results:
    oh = r['reranked_latency_ms'] - r['baseline_latency_ms']
    rows.append({
        'Query Label':              r['query_label'],
        'Baseline (ms)':            f"{r['baseline_latency_ms']:.1f}",
        'Re-Ranking Pipeline (ms)': f"{r['reranked_latency_ms']:.1f}",
        'Overhead (ms)':            f'{oh:.1f}',
        'Slowdown':                 f"{r['reranked_latency_ms']/r['baseline_latency_ms']:.2f}x",
    })

df_lat = pd.DataFrame(rows)
print('LATENCY PROFILING TABLE:')
print(df_lat.to_string(index=False))

b_ms  = [r['baseline_latency_ms']  for r in all_results]
rr_ms = [r['reranked_latency_ms'] for r in all_results]
print(f'\nAvg Baseline:  {np.mean(b_ms):.1f} ms')
print(f'Avg Re-Ranked: {np.mean(rr_ms):.1f} ms')
print(f'Avg Overhead:  +{np.mean(np.array(rr_ms)-np.array(b_ms)):.1f} ms per query')

LATENCY PROFILING TABLE:
                                                  Query Label Baseline (ms) Re-Ranking Pipeline (ms) Overhead (ms) Slowdown
                      General-1: ConMon Overview & Objectives          24.6                    341.0         316.4   13.88x
                     General-2: Significant Change Definition          16.7                    406.2         389.6   24.36x
              F1: Keyword Trap — Incident Reporting Timelines          24.2                    368.1         343.8   15.19x
F2: Exception — Attestation Letter for Small Adaptive Changes          19.6                    383.7         364.1   19.57x
      F3: Negation — Agency ConMon Oversight Responsibilities          19.6                    358.2         338.6   18.27x
              F4: Buried Gold — Table Threshold (60-Day Rule)          22.7                    339.3         316.6   14.97x

Avg Baseline:  21.2 ms
Avg Re-Ranked: 366.1 ms
Avg Overhead:  +344.8 ms per query


## 11. Architecture Comparison: Bi-Encoder vs. Cross-Encoder

| Aspect | Bi-Encoder | Cross-Encoder |
|---|---|---|
| **Input** | Query & passage encoded **independently** | Query + passage **concatenated** into one sequence |
| **Attention** | No cross-attention between Q/P tokens | Full self-attention across **all** token pairs |
| **Score** | Cosine similarity of two embedding vectors | Single relevance logit from the model |
| **Speed** | **O(1)** per query (embeddings pre-built offline) | **O(N)** — one inference forward pass per candidate |
| **Accuracy** | Good approximate relevance | Significantly better — handles negation, synonyms, context |
| **Role** | First-stage filter over millions of docs | Second-stage precise re-ranking over ≤100 candidates |


### Why Not Run Cross-Encoder Over the Whole Corpus?
For a corpus of *N* chunks, running the Cross-Encoder on every chunk for a
single query requires *N* full model forward passes. For this the ConMon Playbook dataset this overhead is manageable, but for larger corpora (e.g., thousands of chunks)
this would take **tens of seconds per query** vs. **milliseconds** for FAISS. At scale
(millions of docs) it becomes completely impractical. The two-stage approach is the
industry-standard solution: the Bi-Encoder cheaply filters to ≤100 candidates,
then the Cross-Encoder applies expensive precision only to that small shortlist.

## 12. Display Benchmark Results

In [51]:
# Create a summary table of the benchmark results
summary_rows = []
for r in all_results:
  summary_rows.append({
      'Query Label': r['query_label'],
      'Baseline Top Chunk': r['baseline_results'][0]['text'][:80].replace(
          '\n', ' '
      )
      + '...',
      'Re-Ranked Top Chunk': r['reranked_results'][0]['text'][:80].replace(
          '\n', ' '
      )
      + '...',
      'Baseline Latency': f"{r['baseline_latency_ms']:.1f} ms",
      'Re-Ranked Latency': f"{r['reranked_latency_ms']:.1f} ms",
  })

df_summary = pd.DataFrame(summary_rows)

print('=' * 80)
print('                        FULL RAG BENCHMARK SUMMARY')
print('=' * 80)
display(df_summary)

                        FULL RAG BENCHMARK SUMMARY


,Query Label,Baseline Top Chunk,Re-Ranked Top Chunk,Baseline Latency,Re-Ranked Latency
0,General-1: ConMon Overview & Objectives,5.0 (2024) ●​ FedRAMP Collaborative ConMon Qu...,ing a plan of action and milestones (POA&Ms) a...,24.6 ms,341.0 ms
1,General-2: Significant Change Definition,on the type of change - implement the signifi...,on the type of change - implement the signifi...,16.7 ms,406.2 ms
2,F1: Keyword Trap — Incident Reporting Timelines,"any emergency inquiry is issued, the CSP must...","cessed, or transmitted by the system as well ...",24.2 ms,368.1 ms
3,F2: Exception — Attestation Letter for Small A...,", except that the SAP will include the assess...",", except that the SAP will include the assess...",19.6 ms,383.7 ms
4,F3: Negation — Agency ConMon Oversight Respons...,) is not responsible for performing ConMon ov...,"fic actions may be taken by the Agency AO, th...",19.6 ms,358.2 ms
5,F4: Buried Gold — Table Threshold (60-Day Rule),act Vulnerabilities Five or more unique vulne...,act Vulnerabilities Five or more unique vulne...,22.7 ms,339.3 ms
